In [ ]:
import sys
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.data.transaction_utils import build_transactions
from src.mln.rule_utils import rules_to_mln_df
from src.rl.action_utils import ACTION_NAMES
from src.rl.ac_model import ActorCriticNet
from src.rl.state_utils import build_initial_rule_state
from src.rl.obs_utils import build_state_vector
from src.rl.reward_utils import compute_reward
from src.rl.simple_env import SimpleRuleEnv

print("all imports ok")

In [ ]:
import sys
import importlib
import random
import pickle
from pathlib import Path
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("seed fixed:", SEED)

In [ ]:
root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.state_utils as state_utils
import src.rl.obs_utils as obs_utils
import src.rl.reward_utils as reward_utils
import src.rl.simple_env as simple_env
import src.rl.ac_model as ac_model
import src.rl.action_utils as action_utils

importlib.reload(state_utils)
importlib.reload(obs_utils)
importlib.reload(reward_utils)
importlib.reload(simple_env)
importlib.reload(ac_model)
importlib.reload(action_utils)

from src.rl.state_utils import build_initial_rule_state
from src.rl.simple_env import SimpleRuleEnv
from src.rl.ac_model import ActorCriticNet
from src.rl.action_utils import ACTION_NAMES

stream_dir = root / "data" / "stream" / "swat"
log_dir = root / "outputs" / "logs"
fig_dir = root / "outputs" / "figures"
model_dir = root / "outputs" / "models"

log_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

print("stream_dir:", stream_dir)

In [ ]:
mixed_rule_pool = pd.read_csv(stream_dir / "mln_mixed_rule_pool.csv")
mixed_state = build_initial_rule_state(mixed_rule_pool)

with open(stream_dir / "a3c_mixed_rule_state.pkl", "wb") as f:
    pickle.dump(mixed_state, f)

print("mixed num_rules:", mixed_state["num_rules"])
print("rule_scores 前5项:", mixed_state["rule_scores"][:5])
print("target_labels 前10项:", mixed_state["target_labels"][:10])
print("saved:", stream_dir / "a3c_mixed_rule_state.pkl")

In [ ]:
with open(stream_dir / "a3c_mixed_rule_state.pkl", "rb") as f:
    mixed_state = pickle.load(f)

# keep 一个攻击规则
env = SimpleRuleEnv(mixed_state, max_steps=1)
env.reset()
_, r_attack_keep, _, info1 = env.step(rule_idx=0, action=0)

# disable 一个攻击规则
env = SimpleRuleEnv(mixed_state, max_steps=1)
env.reset()
_, r_attack_disable, _, info2 = env.step(rule_idx=0, action=1)

# disable 一个正常规则（最后一条）
env = SimpleRuleEnv(mixed_state, max_steps=1)
env.reset()
_, r_normal_disable, _, info3 = env.step(rule_idx=len(mixed_state["target_labels"]) - 1, action=1)

print("r_attack_keep:", r_attack_keep, info1)
print("r_attack_disable:", r_attack_disable, info2)
print("r_normal_disable:", r_normal_disable, info3)

In [ ]:
def train_a3c_mixed(mixed_state, episodes=100, gamma=0.99, lr=1e-3, hidden_dim=64, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = SimpleRuleEnv(mixed_state, max_steps=mixed_state["num_rules"])
    model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    episode_rewards = []

    for episode in range(episodes):
        state = env.reset()
        trajectory = []
        total_reward = 0.0

        for step in range(mixed_state["num_rules"]):
            rule_idx = step

            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits, value = model(state_tensor)
            probs = torch.softmax(logits, dim=-1).squeeze(0)
            action = torch.multinomial(probs, num_samples=1).item()

            next_state, reward, done, info = env.step(rule_idx, action)

            trajectory.append((state, action, reward))
            total_reward += reward
            state = next_state

            if done:
                break

        returns = []
        G = 0.0
        for _, _, r in reversed(trajectory):
            G = r + gamma * G
            returns.insert(0, G)

        states = torch.tensor(np.array([x[0] for x in trajectory], dtype=np.float32))
        actions = torch.tensor(np.array([x[1] for x in trajectory], dtype=np.int64))
        returns = torch.tensor(np.array(returns, dtype=np.float32).reshape(-1, 1))

        logits, values = model(states)
        log_probs = F.log_softmax(logits, dim=-1)
        selected_log_probs = log_probs.gather(1, actions.unsqueeze(1))

        advantages = returns - values
        policy_loss = -(selected_log_probs * advantages.detach()).mean()
        value_loss = F.mse_loss(values, returns)
        loss = policy_loss + 0.5 * value_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        episode_rewards.append(total_reward)

        if (episode + 1) % 10 == 0:
            print(
                f"episode={episode+1}, "
                f"total_reward={total_reward:.4f}, "
                f"policy_loss={policy_loss.detach().item():.4f}, "
                f"value_loss={value_loss.detach().item():.4f}"
            )

    return model, episode_rewards

In [ ]:
with open(stream_dir / "a3c_mixed_rule_state.pkl", "rb") as f:
    mixed_state = pickle.load(f)

model, episode_rewards = train_a3c_mixed(
    mixed_state=mixed_state,
    episodes=100,
    gamma=0.99,
    lr=1e-3,
    hidden_dim=64,
    seed=SEED
)

print("训练完成")
print("最后10个 episode reward:", episode_rewards[-10:])

In [ ]:
torch.save(model.state_dict(), model_dir / "ac_mixed_keep_disable_v1.pth")

reward_df = pd.DataFrame({
    "episode": list(range(1, len(episode_rewards) + 1)),
    "total_reward": episode_rewards
})
reward_df.to_csv(log_dir / "train_rewards_mixed_keep_disable_v1.csv", index=False)

print("saved:", model_dir / "ac_mixed_keep_disable_v1.pth")
print("saved:", log_dir / "train_rewards_mixed_keep_disable_v1.csv")

In [ ]:
df_reward = pd.read_csv(log_dir / "train_rewards_mixed_keep_disable_v1.csv")

plt.figure(figsize=(8, 4))
plt.plot(df_reward["episode"], df_reward["total_reward"])
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Training Reward Curve")
plt.tight_layout()

save_path = fig_dir / "train_rewards_mixed_keep_disable_v1.png"
plt.savefig(save_path, dpi=200)
plt.show()

print("saved:", save_path)
print(df_reward.tail(10))

In [ ]:
def greedy_eval(model, mixed_state):
    env = SimpleRuleEnv(mixed_state, max_steps=mixed_state["num_rules"])

    state = env.reset()
    records = []
    total_reward = 0.0

    for step in range(mixed_state["num_rules"]):
        rule_idx = step

        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        logits, value = model(state_tensor)
        action = torch.argmax(logits, dim=-1).item()

        next_state, reward, done, info = env.step(rule_idx, action)
        total_reward += reward

        rule_info = mixed_state["rule_pool"][rule_idx]
        records.append({
            "step": step + 1,
            "rule_idx": rule_idx,
            "target_label": rule_info["target_label"],
            "formula": rule_info["formula"],
            "action_name": ACTION_NAMES[action],
            "reward": reward,
        })

        state = next_state
        if done:
            break

    eval_df = pd.DataFrame(records)
    return eval_df, total_reward

In [ ]:
eval_df, greedy_total_reward = greedy_eval(model, mixed_state)

print(eval_df[["step", "rule_idx", "target_label", "action_name", "reward"]])

print("\n动作统计：")
print(eval_df["action_name"].value_counts())

print("\n按 target_label 分组统计：")
print(pd.crosstab(eval_df["target_label"], eval_df["action_name"]))

print("\ngreedy total_reward:", greedy_total_reward)

eval_df.to_csv(stream_dir / "greedy_eval_mixed_rules.csv", index=False)

disabled_normal_df = eval_df[
    (eval_df["target_label"] == 0) &
    (eval_df["action_name"] == "disable")
].copy()

disabled_normal_df.to_csv(stream_dir / "disabled_normal_rules.csv", index=False)

print("saved:", stream_dir / "greedy_eval_mixed_rules.csv")
print("saved:", stream_dir / "disabled_normal_rules.csv")
print("被关闭的正常规则数:", len(disabled_normal_df))
print(disabled_normal_df[["rule_idx", "formula", "reward"]])

In [ ]:
attack_keep_rate = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    / (eval_df["target_label"] == 1).sum()
)

normal_disable_rate = (
    ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
    / (eval_df["target_label"] == 0).sum()
)

metrics_df = pd.DataFrame([{
    "attack_keep_rate": attack_keep_rate,
    "normal_disable_rate": normal_disable_rate,
    "greedy_total_reward": greedy_total_reward
}])

metrics_df.to_csv(log_dir / "rule_selection_metrics_v1.csv", index=False)

print("saved:", log_dir / "rule_selection_metrics_v1.csv")
print(metrics_df)

In [ ]:
conf_df = pd.crosstab(
    eval_df["target_label"],
    eval_df["action_name"]
).reindex(index=[0, 1], columns=["keep", "disable"], fill_value=0)

conf_df.index = ["normal_rule", "attack_rule"]
conf_df.to_csv(log_dir / "rule_selection_confusion_v1.csv")

attack_keep = conf_df.loc["attack_rule", "keep"]
normal_disable = conf_df.loc["normal_rule", "disable"]
total_rules = conf_df.values.sum()

selection_accuracy = (attack_keep + normal_disable) / total_rules

acc_df = pd.DataFrame([{
    "attack_keep": attack_keep,
    "normal_disable": normal_disable,
    "total_rules": total_rules,
    "selection_accuracy": selection_accuracy
}])

acc_df.to_csv(log_dir / "rule_selection_accuracy_v1.csv", index=False)

print("saved:", log_dir / "rule_selection_confusion_v1.csv")
print(conf_df)
print("saved:", log_dir / "rule_selection_accuracy_v1.csv")
print(acc_df)

In [ ]:
def run_random_baseline(mixed_state, runs=20, seed=42):
    random.seed(seed)
    records = []

    for run in range(runs):
        env = SimpleRuleEnv(mixed_state, max_steps=mixed_state["num_rules"])
        state = env.reset()
        total_reward = 0.0

        for step in range(mixed_state["num_rules"]):
            rule_idx = step
            action = random.randint(0, 1)  # 0=keep, 1=disable

            next_state, reward, done, info = env.step(rule_idx, action)
            total_reward += reward
            state = next_state

            if done:
                break

        records.append({
            "run": run + 1,
            "total_reward": total_reward
        })

    return pd.DataFrame(records)

random_df = run_random_baseline(mixed_state, runs=20, seed=SEED)

random_df.to_csv(log_dir / "random_policy_baseline_v1.csv", index=False)

print(random_df)
print("random mean reward:", random_df["total_reward"].mean())
print("random std reward:", random_df["total_reward"].std())
print("saved:", log_dir / "random_policy_baseline_v1.csv")

In [ ]:
def run_fixed_policy(mixed_state, fixed_action):
    env = SimpleRuleEnv(mixed_state, max_steps=mixed_state["num_rules"])
    state = env.reset()
    total_reward = 0.0
    records = []

    for step in range(mixed_state["num_rules"]):
        rule_idx = step
        next_state, reward, done, info = env.step(rule_idx, fixed_action)
        total_reward += reward

        rule_info = mixed_state["rule_pool"][rule_idx]
        records.append({
            "rule_idx": rule_idx,
            "target_label": rule_info["target_label"],
            "formula": rule_info["formula"],
            "action": fixed_action,
            "reward": reward
        })

        state = next_state
        if done:
            break

    return total_reward, pd.DataFrame(records)

all_keep_reward, keep_df = run_fixed_policy(mixed_state, fixed_action=0)
all_disable_reward, disable_df = run_fixed_policy(mixed_state, fixed_action=1)

baseline_fixed_df = pd.DataFrame([
    {"method": "All_Keep", "total_reward": all_keep_reward},
    {"method": "All_Disable", "total_reward": all_disable_reward},
])

baseline_fixed_df.to_csv(log_dir / "fixed_policy_baselines_v1.csv", index=False)

print("all_keep_reward:", all_keep_reward)
print("all_disable_reward:", all_disable_reward)
print("saved:", log_dir / "fixed_policy_baselines_v1.csv")
print(baseline_fixed_df)

In [ ]:
all_compare_df = pd.DataFrame([
    {
        "method": "A3C_greedy",
        "total_reward": greedy_total_reward,
        "attack_keep_rate": attack_keep_rate,
        "normal_disable_rate": normal_disable_rate
    },
    {
        "method": "Random_policy_mean",
        "total_reward": random_df["total_reward"].mean(),
        "attack_keep_rate": np.nan,
        "normal_disable_rate": np.nan
    },
    {
        "method": "All_Keep",
        "total_reward": all_keep_reward,
        "attack_keep_rate": np.nan,
        "normal_disable_rate": np.nan
    },
    {
        "method": "All_Disable",
        "total_reward": all_disable_reward,
        "attack_keep_rate": np.nan,
        "normal_disable_rate": np.nan
    }
])

all_compare_df.to_csv(log_dir / "all_method_comparison_v1.csv", index=False)

print("saved:", log_dir / "all_method_comparison_v1.csv")
print(all_compare_df)

In [ ]:
df = pd.read_csv(log_dir / "all_method_comparison_v1.csv")

plt.figure(figsize=(8, 4))
plt.bar(df["method"], df["total_reward"])
plt.ylabel("Total Reward")
plt.title("Method Comparison")
plt.xticks(rotation=20)
plt.tight_layout()

save_path = fig_dir / "all_method_comparison_v1.png"
plt.savefig(save_path, dpi=200)
plt.show()

print("saved:", save_path)
print(df[["method", "total_reward"]])

In [ ]:
plot_df = pd.DataFrame({
    "metric": ["attack_keep_rate", "normal_disable_rate"],
    "value": [attack_keep_rate, normal_disable_rate]
})

plt.figure(figsize=(6, 4))
plt.bar(plot_df["metric"], plot_df["value"])
plt.ylim(0, 1.05)
plt.ylabel("Rate")
plt.title("Rule Selection Metrics")
plt.tight_layout()

save_path = fig_dir / "rule_selection_metrics_v1.png"
plt.savefig(save_path, dpi=200)
plt.show()

print("saved:", save_path)
print(plot_df)

In [ ]:
attack_rules = (eval_df["target_label"] == 1).sum()
normal_rules = (eval_df["target_label"] == 0).sum()

correct_a3c = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    + ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
)

compare_acc_df = pd.DataFrame([
    {
        "method": "A3C_greedy",
        "correct_selected_rules": correct_a3c,
        "total_rules": total_rules,
        "selection_accuracy": correct_a3c / total_rules
    },
    {
        "method": "All_Keep",
        "correct_selected_rules": attack_rules,
        "total_rules": total_rules,
        "selection_accuracy": attack_rules / total_rules
    },
    {
        "method": "All_Disable",
        "correct_selected_rules": normal_rules,
        "total_rules": total_rules,
        "selection_accuracy": normal_rules / total_rules
    }
])

compare_acc_df.to_csv(log_dir / "selection_accuracy_comparison_v1.csv", index=False)

print("saved:", log_dir / "selection_accuracy_comparison_v1.csv")
print(compare_acc_df)

In [ ]:
df = pd.read_csv(log_dir / "selection_accuracy_comparison_v1.csv")

plt.figure(figsize=(6, 4))
plt.bar(df["method"], df["selection_accuracy"])
plt.ylim(0, 1.0)
plt.ylabel("Selection Accuracy")
plt.title("Rule Selection Accuracy Comparison")
plt.tight_layout()

save_path = fig_dir / "selection_accuracy_comparison_v1.png"
plt.savefig(save_path, dpi=200)
plt.show()

print("saved:", save_path)
print(df[["method", "selection_accuracy"]])